In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
class WindowDataset(Dataset):
    def __init__(self, _data, _window):
        # _data: tensor 또는 array의 1차원 데이터 형태
        # _window: 구간의 크기
        self.data = _data
        self.window = _window
        # DataLoader에서 사용 가능한 인덱스의 최대값
        self.n = len(_data) - _window
    
    def __len__(self):
        return self.n
    
    def __getitem__(self, idx):
        # idx: 0 ~ self.n - 1 사이의 정수가 대입 (DataLoader에서 자동으로 대입)
        x = self.data[idx:idx + self.window]
        y = self.data[idx + self.window]
        return x, y

In [ ]:
# RNN 모델 정의

class RNNModel(nn.Module):
    def __init__(self,
                 input_size,
                 hidden_size = 64,
                 num_layers = 1,
                 dropout = 0.0,
                 nonlinearity = 'tanh',
                 bidirectional = False):
        super().__init__()
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            dropout = dropout,
            nonlinearity = nonlinearity,
            bidirectional = bidirectional,
            batch_first = True
        )

        # output_feature가 역방향을 포함한다면 2배로 늘어난다.
        if bidirectional:
            hidden_size *= 2
        
        self.model = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, h_n = self.rnn(x)
        last_hidden = h_n[-1]
        result = self.model(last_hidden)
        return result

In [ ]:
# 모델 학습 시 검증 데이터를 이용하여 모델의 성능을 평가할 수 있도록 검증 데이터 평가 함수

@torch.no_grad()
def evaluate_mse(dataloader, model):
    model.eval()
    total_loss = 0
    total_n = 0
    for x, y in dataloader:
        x = x.float()
        y = y.float()
        pred = model(x)
        loss = nn.MSELoss()(pred, y)
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)
    return total_loss / max(total_n, 1)

In [4]:
df = pd.read_csv('../csv/AAPL.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9715 entries, 0 to 9714
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       9715 non-null   str    
 1   Open       9714 non-null   float64
 2   High       9714 non-null   float64
 3   Low        9714 non-null   float64
 4   Close      9714 non-null   float64
 5   Adj Close  9714 non-null   float64
 6   Volume     9713 non-null   float64
dtypes: float64(6), str(1)
memory usage: 531.4 KB


In [5]:
df.dropna(inplace = True)

In [6]:
df = df[['Date', 'Adj Close']]

In [7]:
values = df[['Adj Close']].values

In [8]:
# 75:25 비율로 학습, 검증 데이터로 분할

split_idx = int(len(values) * 0.75)

train_data = values[:split_idx]
test_data = values[split_idx:]

In [9]:
scaler = MinMaxScaler()
train_sc = scaler.fit_transform(train_data)
test_sc = scaler.transform(test_data)

In [10]:
train_sc = torch.tensor(train_sc, dtype = torch.float32)
test_sc = torch.tensor(test_sc, dtype = torch.float32)

In [11]:
train_ds = WindowDataset(train_sc, _window=60)
test_ds = WindowDataset(test_sc, _window=60)

In [12]:
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

In [13]:
aapl_model = RNNModel(
    input_size = 1
)

In [14]:
criterion = nn.MSELoss()
optimizer = optim.Adam(aapl_model.parameters(), lr=0.01)

In [15]:
# 모델 학습

train_history, test_history = [], []

for epoch in range(20):
    aapl_model.train()
    running, n_seen = 0.0, 0
    for x, y in train_dl:
        x = x.float()
        y = y.float()
        pred = aapl_model(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(aapl_model.parameters(), 1.0)
        optimizer.step()

        running += loss.item() * x.size(0)
        n_seen += x.size(0)
    train_mse = running / n_seen
    test_mse = evaluate_mse(test_dl, aapl_model)
    train_history.append(train_mse)
    test_history.append(test_mse)

    print(f'Epoch {epoch+1}/20 / Train MSE: {round(train_mse, 8)}, Test MSE: {round(test_mse, 8)}')

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([128, 1])) that is different to the input size (torch.Size([60, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (60) must match the size of tensor b (128) at non-singleton dimension 0